# 6. Feature classification: ANOVA vs MI experiments <a id="6"></a>

Compare two ranking strategies for the kinase activation-loop classification pipeline, using a **FoldMason guide-tree** for group-aware train/test splits and scanning jointly over Newick cut height and number of top-ranked features.

| Experiment | Ranking | Matrix |
|---|---|---|
| 1 — ANOVA SUM | `f_classif` | side-chain corr-filtered |
| 2 — Mutual Information | `mutual_info_classif` | side-chain corr-filtered |
| 3 — MI on Cα | `mutual_info_classif` | Cα distances (+ map pairs back to side-chain) |
| 4 — Fixed (k=300, h=12) | MI top-k | SC & Cα × KinCore vs PCA cluster labels |

**August paths:** resolve cwd `corr_filtered_*` / `ca_*` and FoldMason `msa.nw` under `Results/`. Newick is required (no feature-derived substitute).


## Table of contents

- [6.1 Setup & Paths](#setup)
- [6.2 Experiment 1: ANOVA SUM + Newick Tree Parameter Scan](#exp1)
  - [6.2.1 ANOVA parameter scan](#6-2-1-anova-parameter-scan)
  - [6.2.2 Load ANOVA scan + distributions / W/KL](#6-2-2-load-anova-scan-distributions-w-kl)
- [6.3 Experiment 2: Mutual Information + Newick Tree Parameter Scan](#exp2)
- [6.4 Experiment 3: Newick + MI scan with Cα distances](#exp3)
- [6.5 Experiment 4: Fixed parameters (k=300, h=12) — KinCore vs cluster](#exp4)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["11c-ANOVAvsMI"]
  m0["workflow.feature_classification"]
  nb --> m0
  m1["workflow.guide_tree_clusters"]
  nb --> m1
  m2["workflow.pca_analysis"]
  nb --> m2
```


In [ ]:
import os
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif, f_classif

from workflow.feature_classification import FeatureClassification
from workflow.guide_tree_clusters import parse_newick, cluster_leaves_by_height_cut


---

## 6.1 Setup & Paths  <a id="setup"></a>


In [ ]:
SEED = 42

def _first_file(cands):
    for p in cands:
        if p and os.path.isfile(p):
            return p
    return None

def _resolve_newick():
    cands = [
        "Results/activation_segments/multi_aligned_foldmason/msa.nw",
        "Results/multi_aligned_foldmason/msa.nw",
    ]
    cands.extend(sorted(glob("Results/**/msa.nw", recursive=True)))
    for p in cands:
        if os.path.isfile(p):
            return p
    return None

SC_FEATURE_CSV = _first_file([
    "corr_filtered_feature_matrix.csv",
    "filtered_feature_matrix.csv",
])
if SC_FEATURE_CSV is None:
    raise FileNotFoundError("Need corr_filtered_feature_matrix.csv or filtered_feature_matrix.csv")

_sc_prefix = SC_FEATURE_CSV.replace("feature_matrix.csv", "")
SC_LABELS_CSV = _first_file([f"{_sc_prefix}labels.csv", "corr_filtered_labels.csv", "filtered_labels.csv"])
SC_REFERENCE_PKL = _first_file([f"{_sc_prefix}reference_data.pkl", "corr_filtered_reference_data.pkl", "filtered_reference_data.pkl"])
if SC_LABELS_CSV is None:
    raise FileNotFoundError(f"No labels CSV for {SC_FEATURE_CSV}")

CA_FEATURE_CSV = _first_file([
    "ca_corr_filtered_feature_matrix.csv",
    "ca_filtered_feature_matrix.csv",
    "ca_feature_matrix.csv",
])
CA_LABELS_CSV = None
CA_REFERENCE_PKL = None
if CA_FEATURE_CSV:
    _ca_prefix = CA_FEATURE_CSV.replace("feature_matrix.csv", "")
    CA_LABELS_CSV = _first_file([f"{_ca_prefix}labels.csv", "ca_corr_filtered_labels.csv", "ca_filtered_labels.csv", "ca_labels.csv"])
    CA_REFERENCE_PKL = _first_file([f"{_ca_prefix}reference_data.pkl", "ca_corr_filtered_reference_data.pkl", "ca_reference_data.pkl"])

NEWICK_PATH = _resolve_newick()
if NEWICK_PATH is None:
    raise FileNotFoundError(
        "FoldMason Newick (msa.nw) not found under Results/. "
        "Run FoldMason alignment first — do not substitute a feature-derived tree here."
    )

PCA_CLUSTER_LABELS = "cluster_labels_my_analysis_hierarchical.txt"
if not os.path.isfile(PCA_CLUSTER_LABELS):
    alt = "Results/activation_segments/cluster_labels_my_analysis_hierarchical.txt"
    PCA_CLUSTER_LABELS = alt if os.path.isfile(alt) else PCA_CLUSTER_LABELS

KINCORE_CSV = "Results/dunbrack_assignments/kinase_conformation_assignments.csv"

ANOVA_DIR = Path("Results/anova_scan")
MI_DIR = Path("Results/mi_scan")
CA_MI_DIR = Path("Results/ca_mi_scan")
EXP4_DIR = Path("Results/exp4_fixed")
for d in (ANOVA_DIR, MI_DIR, CA_MI_DIR, EXP4_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Feature-count ladder (clamped later to matrix width)
K_VALUES = list(range(300, 2001, 200))
if 2000 not in K_VALUES:
    K_VALUES.append(2000)

print(f"SC matrix  : {SC_FEATURE_CSV}")
print(f"SC labels  : {SC_LABELS_CSV}")
print(f"SC pickle  : {SC_REFERENCE_PKL or '(none)'}")
print(f"Cα matrix  : {CA_FEATURE_CSV or '(missing — Exp3/4 Cα skipped)'}")
print(f"Newick     : {NEWICK_PATH}")
print(f"PCA labels : {PCA_CLUSTER_LABELS if os.path.isfile(PCA_CLUSTER_LABELS) else '(missing)'}")
print(f"KinCore    : {KINCORE_CSV if os.path.isfile(KINCORE_CSV) else '(missing)'}")


In [ ]:
def _bio_csv_for(index, path):
    """Write KinCore (or y-stand-in) bio labels CSV for distribution helpers."""
    if os.path.isfile(KINCORE_CSV):
        from workflow.pca_analysis import ClusterAnalyzer
        ca = ClusterAnalyzer(n_clusters=2)
        bio, _ = ca.load_kincore_labels(list(index), kincore_file=KINCORE_CSV)
        pd.DataFrame({"structure": list(index), "label": np.asarray(bio).astype(int)}).to_csv(path, index=False)
    else:
        # caller should pass y if needed — write zeros as last resort
        pd.DataFrame({"structure": list(index), "label": 0}).to_csv(path, index=False)
    return path

def _split_from_newick(X_df, y, height, Xk=None):
    return FeatureClassification.split_labels_from_newick_guide_tree(
        structure_names=list(X_df.index),
        newick_path=NEWICK_PATH,
        height=height,
        train_size=0.9,
        random_state=42,
        labels=y,
        feature_matrix=Xk if Xk is not None else X_df.values,
    )

def _plot_scan_heatmap(res_df, index_col, col_col, value_col, title, save_path=None):
    pivot = res_df.pivot(index=index_col, columns=col_col, values=value_col)
    fig, ax = plt.subplots(figsize=(10.5, 7.5))
    im = ax.imshow(pivot.values, aspect="auto", origin="lower")
    ax.set_title(title)
    ax.set_xlabel(col_col)
    ax.set_ylabel(index_col)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(x) for x in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([str(x) for x in pivot.index])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

print("Helpers defined")


---

## 6.2 Experiment 1: ANOVA SUM + Newick Tree Parameter Scan  <a id="exp1"></a>

Features ranked by ANOVA F-score (`f_classif`). Group-aware splits from FoldMason Newick cut height.


### 6.2.1 ANOVA parameter scan <a id="6-2-1-anova-parameter-scan"></a>


In [ ]:
_sc = pd.read_csv(SC_FEATURE_CSV, index_col=0)
_k_sc = [k for k in K_VALUES if k <= _sc.shape[1]]
if _sc.shape[1] not in _k_sc and _sc.shape[1] >= 50:
    _k_sc.append(_sc.shape[1])
_k_sc = sorted(set(_k_sc))
print(f"ANOVA k ladder: {_k_sc}")

anova_scan_results = FeatureClassification.run_anova_sum_parameter_scan_with_guide_tree(
    reference_pickle=SC_REFERENCE_PKL,
    feature_matrix_csv=SC_FEATURE_CSV,
    labels_csv=SC_LABELS_CSV,
    newick_path=NEWICK_PATH,
    height_step=2,
    height_min=2,
    height_max=0,
    k_values=_k_sc,
    n_repeats=3,
    train_size=0.9,
    n_estimators=100,
    perm_repeats=50,
    show_plots=True,
    output_dir=str(ANOVA_DIR),
)
print(f"Best ANOVA: h={anova_scan_results['best_h']}, k={anova_scan_results['best_k']}")


### 6.2.2 Load ANOVA scan + distributions / W/KL <a id="6-2-2-load-anova-scan-distributions-w-kl"></a>


In [ ]:
anova_loaded = FeatureClassification.load_anova_scan_results_for_distributions(
    checkpoint_dir=str(ANOVA_DIR / "checkpoints"),
    best_combinations_csv=str(ANOVA_DIR / "best_combinations.csv"),
    reference_pickle=SC_REFERENCE_PKL,
    feature_matrix_csv=SC_FEATURE_CSV,
    labels_csv=SC_LABELS_CSV,
)
# Prefer in-memory scan when present
try:
    anova_loaded = {**anova_loaded, **{k: anova_scan_results[k] for k in
                    ("res_df", "best_h", "best_k", "X_df", "y", "Xk",
                     "feature_labels_all", "gini_mean", "perm_mean") if k in anova_scan_results}}
except NameError:
    pass

best_h_anova = anova_loaded["best_h"]
best_k_anova = anova_loaded["best_k"]
X_df_anova = anova_loaded["X_df"]
y_anova = anova_loaded["y"]
Xk_anova = anova_loaded["Xk"]
feat_lab_anova = anova_loaded["feature_labels_all"]
gini_anova = anova_loaded["gini_mean"]
perm_anova = anova_loaded["perm_mean"]
res_df_anova = anova_loaded.get("res_df")
if res_df_anova is None and (ANOVA_DIR / "metrics_by_height_k.csv").is_file():
    res_df_anova = pd.read_csv(ANOVA_DIR / "metrics_by_height_k.csv")

_plot_scan_heatmap(
    res_df_anova, "height", "k", "accuracy_mean",
    "ANOVA scan — accuracy mean (FoldMason Newick split)",
    save_path=str(ANOVA_DIR / "heatmap_accuracy.png"),
)

bio_anova = str(ANOVA_DIR / "kincore_bio_labels.csv")
_bio_csv_for(X_df_anova.index, bio_anova)
# overwrite with classification labels if KinCore missing
if not os.path.isfile(KINCORE_CSV):
    pd.DataFrame({"structure": list(X_df_anova.index), "label": np.asarray(y_anova).astype(int)}).to_csv(bio_anova, index=False)

split_a, tr_a, te_a, _ = _split_from_newick(X_df_anova, y_anova, best_h_anova, Xk_anova)

if os.path.isfile(PCA_CLUSTER_LABELS):
    dist_anova = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
        X_df=X_df_anova, Xk=Xk_anova, feature_labels_all=feat_lab_anova,
        gini_mean=gini_anova, perm_mean=perm_anova,
        best_h=best_h_anova, best_k=best_k_anova,
        biological_labels_csv=bio_anova,
        pca_cluster_labels_file=PCA_CLUSTER_LABELS,
        split_labels=split_a, train_idx=tr_a, test_idx=te_a,
        n_top=20, title_suffix=f"(ANOVA; h={best_h_anova}, k={best_k_anova})",
    )
    wkl_anova = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_anova_features(
        X_df=X_df_anova, y=y_anova, distribution_plot_results=dist_anova,
        n_anova_features=min(300, X_df_anova.shape[1]), n_bins_kl=31,
    )
    wkl_anova.to_csv(ANOVA_DIR / "wkl_anova.csv", index=False)
    FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
        wkl_anova, best_h=best_h_anova, best_k=best_k_anova,
        n_pool_features=len(wkl_anova), rank_col="anova_rank",
        rank_xlabel="ANOVA F-rank (1 = highest)",
        pool_caption="ANOVA-ranked",
        split_caption="FoldMason Newick group split",
        save_path=str(ANOVA_DIR / "wkl_anova.png"),
    )
else:
    print(f"⚠️  PCA file missing — skip ANOVA violins/WKL")

print("✅ Exp 1 complete")


---

## 6.3 Experiment 2: Mutual Information + Newick Tree Parameter Scan  <a id="exp2"></a>

Same FoldMason split machinery; features ranked by MI (`mutual_info_classif`).


In [ ]:
mi_scan_results = FeatureClassification.run_mi_parameter_scan_with_guide_tree(
    reference_pickle=SC_REFERENCE_PKL,
    feature_matrix_csv=SC_FEATURE_CSV,
    labels_csv=SC_LABELS_CSV,
    newick_path=NEWICK_PATH,
    height_step=2,
    height_min=2,
    height_max=0,
    n_features_values=_k_sc,
    n_repeats=3,
    train_size=0.9,
    n_estimators=100,
    perm_repeats=50,
    show_plots=True,
    output_dir=str(MI_DIR),
)
print(f"Best MI: h={mi_scan_results['best_h']}, n={mi_scan_results['best_n']}")


In [ ]:
mi_loaded = FeatureClassification.load_mi_scan_results_for_distributions(
    checkpoint_dir=str(MI_DIR / "checkpoints"),
    best_combinations_csv=str(MI_DIR / "best_combinations.csv"),
    reference_pickle=SC_REFERENCE_PKL,
    feature_matrix_csv=SC_FEATURE_CSV,
    labels_csv=SC_LABELS_CSV,
)
try:
    mi_loaded = {**mi_loaded, **{k: mi_scan_results[k] for k in
                ("res_df", "best_h", "best_n", "X_df", "y", "Xk",
                 "feature_labels_all", "gini_mean", "perm_mean") if k in mi_scan_results}}
except NameError:
    pass

best_h_mi = mi_loaded["best_h"]
best_n_mi = mi_loaded["best_n"]
X_df_mi = mi_loaded["X_df"]
y_mi = mi_loaded["y"]
Xk_mi = mi_loaded["Xk"]
feat_lab_mi = mi_loaded["feature_labels_all"]
gini_mi = mi_loaded["gini_mean"]
perm_mi = mi_loaded["perm_mean"]
res_df_mi = mi_loaded.get("res_df")
if res_df_mi is None and (MI_DIR / "metrics_by_height_n.csv").is_file():
    res_df_mi = pd.read_csv(MI_DIR / "metrics_by_height_n.csv")

# Ensure balanced_accuracy columns exist (no-op if already present from scan)
if res_df_mi is not None and "balanced_accuracy_mean" not in res_df_mi.columns:
    res_df_mi = FeatureClassification.add_balanced_accuracy_to_mi_scan_metrics(
        res_df_mi,
        reference_pickle=SC_REFERENCE_PKL,
        feature_matrix_csv=SC_FEATURE_CSV,
        labels_csv=SC_LABELS_CSV,
        newick_path=NEWICK_PATH,
        n_repeats=3,
    )
    res_df_mi.to_csv(MI_DIR / "metrics_by_height_n.csv", index=False)

_plot_scan_heatmap(
    res_df_mi, "height", "n_features", "accuracy_mean",
    "MI scan — accuracy mean (FoldMason Newick split)",
    save_path=str(MI_DIR / "heatmap_accuracy.png"),
)

bio_mi = str(MI_DIR / "kincore_bio_labels.csv")
_bio_csv_for(X_df_mi.index, bio_mi)
if not os.path.isfile(KINCORE_CSV):
    pd.DataFrame({"structure": list(X_df_mi.index), "label": np.asarray(y_mi).astype(int)}).to_csv(bio_mi, index=False)

split_m, tr_m, te_m, _ = _split_from_newick(X_df_mi, y_mi, best_h_mi, Xk_mi)

if os.path.isfile(PCA_CLUSTER_LABELS):
    dist_mi = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
        X_df=X_df_mi, Xk=Xk_mi, feature_labels_all=feat_lab_mi,
        gini_mean=gini_mi, perm_mean=perm_mi,
        best_h=best_h_mi, best_k=best_n_mi,
        biological_labels_csv=bio_mi,
        pca_cluster_labels_file=PCA_CLUSTER_LABELS,
        split_labels=split_m, train_idx=tr_m, test_idx=te_m,
        n_top=20, title_suffix=f"(MI; h={best_h_mi}, n={best_n_mi})",
    )
    wkl_mi = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_mi_features(
        X_df=X_df_mi, y=y_mi, distribution_plot_results=dist_mi,
        n_mi_features=min(300, X_df_mi.shape[1]), n_bins_kl=31,
    )
    wkl_mi.to_csv(MI_DIR / "wkl_mi.csv", index=False)
    FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
        wkl_mi, best_h=best_h_mi, best_k=best_n_mi,
        n_pool_features=len(wkl_mi), rank_col="mi_rank",
        rank_xlabel="MI rank (1 = highest)",
        pool_caption="MI-ranked",
        split_caption="FoldMason Newick group split",
        save_path=str(MI_DIR / "wkl_mi.png"),
    )

# Optional: Cα violins at SC MI column indices (same index array when columns align)
if CA_FEATURE_CSV and os.path.isfile(PCA_CLUSTER_LABELS):
    X_ca = pd.read_csv(CA_FEATURE_CSV, index_col=0)
    shared = [s for s in X_df_mi.index if s in X_ca.index]
    if len(shared) >= 20:
        X_ca = X_ca.loc[shared]
        # reuse MI-selected column names that exist in Cα
        mi_cols = list(pd.DataFrame(Xk_mi, index=X_df_mi.index, columns=[str(c) for c in range(Xk_mi.shape[1])]).columns)
        print(f"Cα overlap structures: {len(shared)} (index-based MI→Cα overlay skipped if columns differ)")
    else:
        print("Insufficient SC∩Cα overlap for Exp2 Cα overlay")

print("✅ Exp 2 complete")


---

## 6.4 Experiment 3: Newick + MI scan with Cα distances  <a id="exp3"></a>

MI parameter scan on the Cα matrix, then map top Cα MI pairs onto side-chain distances for comparison.


In [ ]:
if CA_FEATURE_CSV is None or CA_LABELS_CSV is None:
    print("⚠️  Cα matrix/labels missing — skipping Exp 3")
else:
    _ca = pd.read_csv(CA_FEATURE_CSV, index_col=0)
    _k_ca = [k for k in K_VALUES if k <= _ca.shape[1]]
    if _ca.shape[1] not in _k_ca and _ca.shape[1] >= 50:
        _k_ca.append(_ca.shape[1])
    _k_ca = sorted(set(_k_ca))
    print(f"Cα MI n_features ladder: {_k_ca}")

    ca_mi_scan = FeatureClassification.run_mi_parameter_scan_with_guide_tree(
        reference_pickle=CA_REFERENCE_PKL,
        feature_matrix_csv=CA_FEATURE_CSV,
        labels_csv=CA_LABELS_CSV,
        newick_path=NEWICK_PATH,
        height_step=2,
        height_min=2,
        height_max=0,
        n_features_values=_k_ca,
        n_repeats=3,
        train_size=0.9,
        n_estimators=100,
        perm_repeats=50,
        show_plots=True,
        output_dir=str(CA_MI_DIR),
    )
    print(f"Best Cα MI: h={ca_mi_scan['best_h']}, n={ca_mi_scan['best_n']}")


In [ ]:
if CA_FEATURE_CSV is None or CA_LABELS_CSV is None:
    print("⚠️  Skipping Exp 3 load/plots")
else:
    ca_mi_loaded = FeatureClassification.load_mi_scan_results_for_distributions(
        checkpoint_dir=str(CA_MI_DIR / "checkpoints"),
        best_combinations_csv=str(CA_MI_DIR / "best_combinations.csv"),
        reference_pickle=CA_REFERENCE_PKL,
        feature_matrix_csv=CA_FEATURE_CSV,
        labels_csv=CA_LABELS_CSV,
    )
    try:
        ca_mi_loaded = {**ca_mi_loaded, **{k: ca_mi_scan[k] for k in
                        ("res_df", "best_h", "best_n", "X_df", "y", "Xk",
                         "feature_labels_all", "gini_mean", "perm_mean") if k in ca_mi_scan}}
    except NameError:
        pass

    best_h_ca = ca_mi_loaded["best_h"]
    best_n_ca = ca_mi_loaded["best_n"]
    X_df_ca3 = ca_mi_loaded["X_df"]
    y_ca3 = ca_mi_loaded["y"]
    Xk_ca3 = ca_mi_loaded["Xk"]
    feat_lab_ca3 = ca_mi_loaded["feature_labels_all"]
    gini_ca3 = ca_mi_loaded["gini_mean"]
    perm_ca3 = ca_mi_loaded["perm_mean"]

    bio_ca = str(CA_MI_DIR / "kincore_bio_labels.csv")
    _bio_csv_for(X_df_ca3.index, bio_ca)
    if not os.path.isfile(KINCORE_CSV):
        pd.DataFrame({"structure": list(X_df_ca3.index), "label": np.asarray(y_ca3).astype(int)}).to_csv(bio_ca, index=False)

    split_ca, tr_ca, te_ca, _ = _split_from_newick(X_df_ca3, y_ca3, best_h_ca, Xk_ca3)

    if os.path.isfile(PCA_CLUSTER_LABELS):
        dist_ca3 = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
            X_df=X_df_ca3, Xk=Xk_ca3, feature_labels_all=feat_lab_ca3,
            gini_mean=gini_ca3, perm_mean=perm_ca3,
            best_h=best_h_ca, best_k=best_n_ca,
            biological_labels_csv=bio_ca,
            pca_cluster_labels_file=PCA_CLUSTER_LABELS,
            split_labels=split_ca, train_idx=tr_ca, test_idx=te_ca,
            n_top=20, title_suffix=f"(Cα MI; h={best_h_ca}, n={best_n_ca})",
        )
        wkl_ca3 = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_mi_features(
            X_df=X_df_ca3, y=y_ca3, distribution_plot_results=dist_ca3,
            n_mi_features=min(int(best_n_ca), X_df_ca3.shape[1]), n_bins_kl=31,
        )
        wkl_ca3.to_csv(CA_MI_DIR / "wkl_ca_mi.csv", index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl_ca3, best_h=best_h_ca, best_k=best_n_ca,
            n_pool_features=len(wkl_ca3), rank_col="mi_rank",
            rank_xlabel="Cα MI rank (1 = highest)",
            pool_caption="MI-ranked Cα",
            save_path=str(CA_MI_DIR / "wkl_ca_mi.png"),
        )

    # Map Cα MI-ranked pair keys onto side-chain matrix columns
    def _pair_key(name):
        return FeatureClassification._normalize_pair_key(name)

    # Rebuild full MI ranking on Cα to get ordered column names
    X_ca_imp = FeatureClassification._impute_feature_matrix(X_df_ca3)
    mi_ca = mutual_info_classif(X_ca_imp, y_ca3, random_state=42)
    mi_ca = np.nan_to_num(mi_ca, nan=0.0)
    order_ca = np.argsort(mi_ca)[::-1][: int(best_n_ca)]
    ca_pair_keys = [_pair_key(X_df_ca3.columns[j]) for j in order_ca]
    ca_pair_keys = [k for k in ca_pair_keys if k]

    X_sc = pd.read_csv(SC_FEATURE_CSV, index_col=0)
    sc_key_to_col = {}
    for j, c in enumerate(X_sc.columns):
        k = _pair_key(c)
        if k and k not in sc_key_to_col:
            sc_key_to_col[k] = j
    mapped_cols = [sc_key_to_col[k] for k in ca_pair_keys if k in sc_key_to_col]
    print(f"Mapped {len(mapped_cols)} / {len(ca_pair_keys)} Cα MI pairs onto side-chain columns")

    if mapped_cols and os.path.isfile(PCA_CLUSTER_LABELS):
        shared = [s for s in X_df_ca3.index if s in X_sc.index]
        X_sc_m = X_sc.loc[shared]
        y_sc_m = np.asarray(y_ca3)[[list(X_df_ca3.index).index(s) for s in shared]]
        Xk_sc_m = FeatureClassification._impute_feature_matrix(X_sc_m)[:, mapped_cols]
        feat_sc_m = [str(X_sc_m.columns[j]) for j in mapped_cols]
        # synthetic importances = reverse rank order
        gini_sc_m = np.linspace(1.0, 0.1, len(mapped_cols))
        perm_sc_m = gini_sc_m.copy()
        split_scm, tr_scm, te_scm, _ = _split_from_newick(X_sc_m, y_sc_m, best_h_ca, Xk_sc_m)
        bio_scm = str(CA_MI_DIR / "kincore_bio_labels_sc_mapped.csv")
        _bio_csv_for(X_sc_m.index, bio_scm)
        if not os.path.isfile(KINCORE_CSV):
            pd.DataFrame({"structure": list(X_sc_m.index), "label": y_sc_m.astype(int)}).to_csv(bio_scm, index=False)
        dist_scm = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
            X_df=X_sc_m, Xk=Xk_sc_m, feature_labels_all=feat_sc_m,
            gini_mean=gini_sc_m, perm_mean=perm_sc_m,
            best_h=best_h_ca, best_k=len(mapped_cols),
            biological_labels_csv=bio_scm,
            pca_cluster_labels_file=PCA_CLUSTER_LABELS,
            split_labels=split_scm, train_idx=tr_scm, test_idx=te_scm,
            n_top=min(20, len(mapped_cols)),
            title_suffix=f"(SC @ Cα MI ranks; h={best_h_ca})",
        )
        # W/KL on mapped SC columns using MI-style ranking of the mapped subset
        # Build a mini X_df of mapped cols only for MI WKL helper
        X_map_df = X_sc_m.iloc[:, mapped_cols].copy()
        X_map_df.columns = feat_sc_m
        dist_scm_w = dict(dist_scm)
        dist_scm_w["split_labels"] = split_scm
        wkl_scm = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_mi_features(
            X_df=X_map_df, y=y_sc_m, distribution_plot_results=dist_scm_w,
            n_mi_features=len(mapped_cols), n_bins_kl=31,
        )
        wkl_scm.to_csv(CA_MI_DIR / "wkl_sc_for_ca_mi_features.csv", index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl_scm, best_h=best_h_ca, best_k=len(mapped_cols),
            n_pool_features=len(wkl_scm), rank_col="mi_rank",
            rank_xlabel="Cα MI rank → side-chain distances",
            pool_caption="Cα MI rank → SC distances",
            save_path=str(CA_MI_DIR / "wkl_sc_for_ca_mi.png"),
        )

    print("✅ Exp 3 complete")


---

## 6.5 Experiment 4: Fixed parameters (k=300, h=12) — KinCore vs cluster  <a id="exp4"></a>

Train RFs at a fixed operating point on Cα and side-chain matrices, predicting KinCore activation and PCA cluster labels.


In [ ]:
EXP4_H = 12
EXP4_K = 300
EXP4_SEEDS = [42, 43, 44]
N_ESTIMATORS = 100

# Clamp height to tree if needed
_tree = parse_newick(Path(NEWICK_PATH).read_text())
_, _max_h = cluster_leaves_by_height_cut(_tree, cut_height=1e9)
if float(EXP4_H) >= float(_max_h):
    EXP4_H = max(1, int(np.floor(_max_h - 1e-9)))
    print(f"⚠️  Clamped EXP4_H to {EXP4_H} (tree max ≈ {_max_h})")

def _groups_at_h(structure_names, height):
    leaf_to_cluster, _ = cluster_leaves_by_height_cut(_tree, cut_height=float(height))
    next_missing = (max(leaf_to_cluster.values()) + 1) if leaf_to_cluster else 1
    groups = []
    for nm in structure_names:
        g = leaf_to_cluster.get(nm)
        if g is None:
            g = next_missing
            next_missing += 1
        groups.append(g)
    return np.asarray(groups)

def _mi_topk(X_df, y, k):
    X_imp = FeatureClassification._impute_feature_matrix(X_df)
    scores = mutual_info_classif(X_imp, y, random_state=42)
    scores = np.nan_to_num(scores, nan=0.0)
    k_use = min(int(k), X_imp.shape[1])
    idx = np.sort(np.argsort(scores)[::-1][:k_use])
    return X_imp, idx, scores

def _train_rf_repeats(Xn, y, pairs, names, groups, seeds, n_estimators=100):
    accs, precs, ginis, perms, labels = [], [], [], [], None
    for seed in seeds:
        clf = FeatureClassification(
            feature_matrix=Xn, labels=y, unique_pairs=pairs,
            structure_names=names,
        )
        clf.split_data(train_size=0.9, random_state=seed, groups=groups)
        clf.train_model(n_estimators=n_estimators, random_state=seed, n_jobs=-1)
        m = clf.evaluate_model(show_metrics=False)
        accs.append(m["accuracy"])
        precs.append(m["precision"])
        g, _, _ = clf.compute_feature_importances()
        ginis.append(g)
        pr = clf.compute_permutation_importances(n_repeats=10, random_state=seed, n_jobs=-1)
        perms.append(pr.importances_mean)
        if labels is None:
            labels = [f"{a}-{b}" if isinstance(a, int) else str(a) for a, b in pairs]
    return {
        "accuracy_mean": float(np.mean(accs)),
        "accuracy_std": float(np.std(accs)),
        "precision_mean": float(np.mean(precs)),
        "precision_std": float(np.std(precs)),
        "gini_mean": np.mean(np.stack(ginis), axis=0),
        "perm_mean": np.mean(np.stack(perms), axis=0),
        "feature_labels": labels,
        "Xk": Xn,
        "y": y,
    }

def _pca_labels_for(index):
    if not os.path.isfile(PCA_CLUSTER_LABELS):
        return None
    return FeatureClassification._load_pca_cluster_labels(PCA_CLUSTER_LABELS, index)

def _kincore_labels_for(index):
    if not os.path.isfile(KINCORE_CSV):
        return None
    from workflow.pca_analysis import ClusterAnalyzer
    bio, _ = ClusterAnalyzer(n_clusters=2).load_kincore_labels(list(index), kincore_file=KINCORE_CSV)
    return np.asarray(bio, dtype=float)

exp4_rows = []
exp4_models = {}

# Side-chain
X_sc4 = pd.read_csv(SC_FEATURE_CSV, index_col=0)
y_sc_file = pd.read_csv(SC_LABELS_CSV)
if "structure" in y_sc_file.columns:
    y_sc_file = y_sc_file.set_index("structure").loc[X_sc4.index]
y_sc_cls = y_sc_file["label"].values
groups_sc = _groups_at_h(list(X_sc4.index), EXP4_H)

for tag, y_target in [
    ("sc_kincore", _kincore_labels_for(X_sc4.index)),
    ("sc_cluster", _pca_labels_for(X_sc4.index)),
]:
    if y_target is None or np.sum(np.isfinite(y_target)) < 20:
        print(f"⚠️  Skip {tag}: labels missing/insufficient")
        continue
    mask = np.isfinite(y_target)
    X_sub = X_sc4.loc[np.asarray(X_sc4.index)[mask]]
    y_sub = y_target[mask].astype(int)
    g_sub = groups_sc[mask]
    X_imp, idx, _ = _mi_topk(X_sub, y_sub, EXP4_K)
    pairs = []
    for j in idx:
        c = str(X_sub.columns[j])
        parts = c.split("-")
        pairs.append((int(parts[0]), int(parts[1])) if len(parts) == 2 and parts[0].isdigit() else (c, c))
    res = _train_rf_repeats(X_imp[:, idx], y_sub, pairs, list(X_sub.index), g_sub, EXP4_SEEDS, N_ESTIMATORS)
    exp4_models[tag] = {**res, "X_df": X_sub, "idx": idx, "groups": g_sub, "height": EXP4_H}
    exp4_rows.append({"tag": tag, **{k: res[k] for k in ("accuracy_mean", "accuracy_std", "precision_mean", "precision_std")}})
    print(f"{tag}: acc={res['accuracy_mean']:.3f}±{res['accuracy_std']:.3f}")

# Cα
if CA_FEATURE_CSV and CA_LABELS_CSV:
    X_ca4 = pd.read_csv(CA_FEATURE_CSV, index_col=0)
    groups_ca = _groups_at_h(list(X_ca4.index), EXP4_H)
    for tag, y_target in [
        ("ca_kincore", _kincore_labels_for(X_ca4.index)),
        ("ca_cluster", _pca_labels_for(X_ca4.index)),
    ]:
        if y_target is None or np.sum(np.isfinite(y_target)) < 20:
            print(f"⚠️  Skip {tag}: labels missing/insufficient")
            continue
        mask = np.isfinite(y_target)
        X_sub = X_ca4.loc[np.asarray(X_ca4.index)[mask]]
        y_sub = y_target[mask].astype(int)
        g_sub = groups_ca[mask]
        X_imp, idx, _ = _mi_topk(X_sub, y_sub, EXP4_K)
        pairs = []
        for j in idx:
            c = str(X_sub.columns[j])
            parts = c.split("-")
            pairs.append((int(parts[0]), int(parts[1])) if len(parts) == 2 and parts[0].isdigit() else (c, c))
        res = _train_rf_repeats(X_imp[:, idx], y_sub, pairs, list(X_sub.index), g_sub, EXP4_SEEDS, N_ESTIMATORS)
        exp4_models[tag] = {**res, "X_df": X_sub, "idx": idx, "groups": g_sub, "height": EXP4_H}
        exp4_rows.append({"tag": tag, **{k: res[k] for k in ("accuracy_mean", "accuracy_std", "precision_mean", "precision_std")}})
        print(f"{tag}: acc={res['accuracy_mean']:.3f}±{res['accuracy_std']:.3f}")
else:
    print("⚠️  Cα missing — Exp4 Cα arms skipped")

perf4 = pd.DataFrame(exp4_rows)
perf4.to_csv(EXP4_DIR / "summary_metrics.csv", index=False)
print(perf4.to_string(index=False))
print("✅ Exp 4.1 training complete")


In [ ]:
# Exp 4.2–4.3: distributions + W/KL for each trained arm (cluster targets only need PCA file)
for tag, model in exp4_models.items():
    print("\n" + "=" * 60)
    print(tag)
    X_df = model["X_df"]
    Xk = model["Xk"]
    y = model["y"]
    split, tr, te, _ = FeatureClassification.split_labels_from_newick_guide_tree(
        structure_names=list(X_df.index),
        newick_path=NEWICK_PATH,
        height=model["height"],
        train_size=0.9,
        random_state=42,
        labels=y,
        feature_matrix=Xk,
    )
    bio = str(EXP4_DIR / f"bio_{tag}.csv")
    pd.DataFrame({"structure": list(X_df.index), "label": np.asarray(y).astype(int)}).to_csv(bio, index=False)

    if "cluster" in tag and os.path.isfile(PCA_CLUSTER_LABELS):
        dist = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
            X_df=X_df, Xk=Xk, feature_labels_all=model["feature_labels"],
            gini_mean=model["gini_mean"], perm_mean=model["perm_mean"],
            best_h=model["height"], best_k=Xk.shape[1],
            biological_labels_csv=bio,
            pca_cluster_labels_file=PCA_CLUSTER_LABELS,
            split_labels=split, train_idx=tr, test_idx=te,
            n_top=20, title_suffix=f"(Exp4 {tag}; h={model['height']}, k={Xk.shape[1]})",
        )
        wkl = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_mi_features(
            X_df=X_df, y=y, distribution_plot_results=dist,
            n_mi_features=min(300, X_df.shape[1]), n_bins_kl=31,
        )
        wkl.to_csv(EXP4_DIR / f"wkl_{tag}.csv", index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl, best_h=model["height"], best_k=Xk.shape[1],
            n_pool_features=len(wkl), rank_col="mi_rank",
            pool_caption=f"Exp4 {tag}",
            save_path=str(EXP4_DIR / f"wkl_{tag}.png"),
        )
    elif "kincore" in tag:
        dist = FeatureClassification.plot_top_feature_distributions_by_activation(
            X_df=X_df, Xk=Xk, feature_labels_all=model["feature_labels"],
            gini_mean=model["gini_mean"], perm_mean=model["perm_mean"],
            best_h=model["height"], best_k=Xk.shape[1],
            biological_labels_csv=bio,
            split_labels=split, train_idx=tr, test_idx=te,
            n_top=20, title_suffix=f"(Exp4 {tag}; h={model['height']}, k={Xk.shape[1]})",
        )
        wkl = FeatureClassification.compute_active_vs_inactive_wasserstein_kl_for_rf_features(
            X_df=X_df, distribution_plot_results=dist,
            importance=model["gini_mean"], rank_col="mdi_rank",
            n_features=min(300, X_df.shape[1]), n_bins_kl=31,
        )
        wkl.to_csv(EXP4_DIR / f"wkl_{tag}.csv", index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl, best_h=model["height"], best_k=Xk.shape[1],
            n_pool_features=len(wkl), rank_col="mdi_rank",
            comparison_label="inactive vs active",
            pool_caption=f"Exp4 {tag}",
            save_path=str(EXP4_DIR / f"wkl_{tag}.png"),
        )
    else:
        print(f"Skip plots for {tag}")

print("=" * 60)
print("✅ 11c FEATURE SELECTION EXPERIMENTS COMPLETE (Exp 1–4)")
print("=" * 60)
